In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import joblib

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
PROJECT_ROOT = Path(r"C:\Case Study Trees\HomeCreditStudy")

SPLIT_DIR = PROJECT_ROOT / "data" / "splits"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(SPLIT_DIR / "train.csv")
valid_df = pd.read_csv(SPLIT_DIR / "valid.csv")
test_df = pd.read_csv(SPLIT_DIR / "test.csv")

print("Training:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

Training: (215256, 122)
Validation: (46127, 122)
Test: (46127, 122)


In [3]:
y_train = train_df["TARGET"].copy()
y_valid = valid_df["TARGET"].copy()
y_test = test_df["TARGET"].copy()


train_ids = train_df["SK_ID_CURR"].copy()
valid_ids = valid_df["SK_ID_CURR"].copy()
test_ids = test_df["SK_ID_CURR"].copy()


X_train = train_df.drop(columns=["TARGET", "SK_ID_CURR"]).copy()
X_valid = valid_df.drop(columns=["TARGET", "SK_ID_CURR"]).copy()
X_test = test_df.drop(columns=["TARGET", "SK_ID_CURR"]).copy()

print("Training features:", X_train.shape)
print("Validation features:", X_valid.shape)
print("Test features:", X_test.shape)

Training features: (215256, 120)
Validation features: (46127, 120)
Test features: (46127, 120)


In [4]:
for data in [X_train, X_valid, X_test]:
    data["DAYS_EMPLOYED_WAS_SENTINEL"] = (
        data["DAYS_EMPLOYED"].eq(365243).astype(int)
    )

    data["DAYS_EMPLOYED"] = data["DAYS_EMPLOYED"].replace(
        365243, np.nan
    )

print("Training sentinel indicator:")
print(X_train["DAYS_EMPLOYED_WAS_SENTINEL"].value_counts())

print("\nRemaining sentinel values:")
print(X_train["DAYS_EMPLOYED"].eq(365243).sum())

Training sentinel indicator:
DAYS_EMPLOYED_WAS_SENTINEL
0    176638
1     38618
Name: count, dtype: int64

Remaining sentinel values:
0


In [8]:
for data in [X_train, X_valid, X_test]:
    if "DAYS_BIRTH" in data.columns:
        data["AGE_YEARS"] = -data["DAYS_BIRTH"] / 365.25
        data.drop(columns="DAYS_BIRTH", inplace=True)

    if "DAYS_EMPLOYED" in data.columns:
        data["EMPLOYED_YEARS"] = -data["DAYS_EMPLOYED"] / 365.25
        data.drop(columns="DAYS_EMPLOYED", inplace=True)

display(X_train[["AGE_YEARS", "EMPLOYED_YEARS"]].head())

,AGE_YEARS,EMPLOYED_YEARS
0,42.880219,14.885695
1,49.130732,16.553046
2,38.737851,0.490075
3,61.453799,NaN
4,27.178645,6.735113


In [9]:
display(
    X_train[
        ["EMPLOYED_YEARS", "DAYS_EMPLOYED_WAS_SENTINEL"]
    ].head(10).round(4)
)

,EMPLOYED_YEARS,DAYS_EMPLOYED_WAS_SENTINEL
0,14.8857,0
1,16.5530,0
2,0.4901,0
3,NaN,1
4,6.7351,0
5,5.2183,0
6,17.8946,0
7,2.7789,0
8,21.1362,0
9,2.3901,0


In [10]:
for data in [X_train, X_valid, X_test]:
    income = data["AMT_INCOME_TOTAL"].replace(0, np.nan)
    goods_price = data["AMT_GOODS_PRICE"].replace(0, np.nan)
    family_size = data["CNT_FAM_MEMBERS"].replace(0, np.nan)

    data["CREDIT_INCOME_RATIO"] = data["AMT_CREDIT"] / income
    data["ANNUITY_INCOME_RATIO"] = data["AMT_ANNUITY"] / income
    data["CREDIT_GOODS_RATIO"] = data["AMT_CREDIT"] / goods_price
    data["INCOME_PER_FAMILY_MEMBER"] = income / family_size

new_features = [
    "CREDIT_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO",
    "CREDIT_GOODS_RATIO",
    "INCOME_PER_FAMILY_MEMBER"
]

display(X_train[new_features].head())

,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO,CREDIT_GOODS_RATIO,INCOME_PER_FAMILY_MEMBER
0,2.000000,0.135300,1.0000,45000.0
1,1.500000,0.075000,1.0000,112500.0
2,2.352941,0.117647,1.0000,38250.0
3,9.816000,0.287829,1.1452,157500.0
4,2.460870,0.258391,1.1320,103500.0


In [ ]:
#Function to do it again and again

def create_features(data):
    data = data.copy()

    data["DAYS_EMPLOYED_WAS_SENTINEL"] = (
        data["DAYS_EMPLOYED"].eq(365243).astype(int)
    )

    data["DAYS_EMPLOYED"] = data["DAYS_EMPLOYED"].replace(
        365243, np.nan
    )

 
    data["AGE_YEARS"] = -data["DAYS_BIRTH"] / 365.25
    data["EMPLOYED_YEARS"] = -data["DAYS_EMPLOYED"] / 365.25

    income = data["AMT_INCOME_TOTAL"].replace(0, np.nan)
    goods_price = data["AMT_GOODS_PRICE"].replace(0, np.nan)
    family_size = data["CNT_FAM_MEMBERS"].replace(0, np.nan)

    data["CREDIT_INCOME_RATIO"] = data["AMT_CREDIT"] / income
    data["ANNUITY_INCOME_RATIO"] = data["AMT_ANNUITY"] / income
    data["CREDIT_GOODS_RATIO"] = data["AMT_CREDIT"] / goods_price
    data["INCOME_PER_FAMILY_MEMBER"] = income / family_size

    data.drop(columns=["DAYS_BIRTH", "DAYS_EMPLOYED"], inplace=True)

    return data.replace([np.inf, -np.inf], np.nan)

In [11]:
for data in [X_train, X_valid, X_test]:
    for column in ["CODE_GENDER", "ORGANIZATION_TYPE"]:
        data[column] = data[column].replace("XNA", np.nan)

print("Remaining XNA values in training:")

print(
    X_train[["CODE_GENDER", "ORGANIZATION_TYPE"]]
    .eq("XNA")
    .sum()
)

Remaining XNA values in training:
CODE_GENDER          0
ORGANIZATION_TYPE    0
dtype: int64


In [12]:
columns_to_drop = []

for column in X_train.columns:
    if column.endswith("_AVG"):
        feature = column[:-4]

        for suffix in ["_MEDI", "_MODE"]:
            other_column = feature + suffix

            if other_column in X_train.columns:
                correlation = X_train[column].corr(
                    X_train[other_column]
                )

                same_missing = X_train[column].isna().equals(
                    X_train[other_column].isna()
                )

                if abs(correlation) > 0.95 and same_missing:
                    columns_to_drop.append(other_column)

print("Number of columns to remove:", len(columns_to_drop))
print(columns_to_drop)

Number of columns to remove: 28
['APARTMENTS_MEDI', 'APARTMENTS_MODE', 'BASEMENTAREA_MEDI', 'BASEMENTAREA_MODE', 'YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BEGINEXPLUATATION_MODE', 'YEARS_BUILD_MEDI', 'YEARS_BUILD_MODE', 'COMMONAREA_MEDI', 'COMMONAREA_MODE', 'ELEVATORS_MEDI', 'ELEVATORS_MODE', 'ENTRANCES_MEDI', 'ENTRANCES_MODE', 'FLOORSMAX_MEDI', 'FLOORSMAX_MODE', 'FLOORSMIN_MEDI', 'FLOORSMIN_MODE', 'LANDAREA_MEDI', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MEDI', 'LIVINGAREA_MODE', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MEDI', 'NONLIVINGAREA_MODE']


In [13]:
numeric_columns = X_train.select_dtypes(include="number").columns.tolist()
categorical_columns = X_train.select_dtypes(include="object").columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True,
        keep_empty_features=True
    ))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Missing",
        keep_empty_features=True
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
        dtype=np.float32
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

print("Numeric columns:", len(numeric_columns))
print("Categorical columns:", len(categorical_columns))

Numeric columns: 109
Categorical columns: 16


In [14]:
X_train_processed = preprocessor.fit_transform(X_train)

X_valid_processed = preprocessor.transform(X_valid)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_valid_processed = pd.DataFrame(
    X_valid_processed,
    columns=feature_names,
    index=X_valid.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

print("Training:", X_train_processed.shape)
print("Validation:", X_valid_processed.shape)
print("Test:", X_test_processed.shape)

print("\nMissing values after preprocessing:")
print("Training:", X_train_processed.isna().sum().sum())
print("Validation:", X_valid_processed.isna().sum().sum())
print("Test:", X_test_processed.isna().sum().sum())

Training: (215256, 320)
Validation: (46127, 320)
Test: (46127, 320)

Missing values after preprocessing:
Training: 0
Validation: 0
Test: 0


In [15]:
X_train_filtered = X_train.drop(columns=columns_to_drop).copy()
X_valid_filtered = X_valid.drop(columns=columns_to_drop).copy()
X_test_filtered = X_test.drop(columns=columns_to_drop).copy()

print("Engineered features:", X_train.shape[1])
print("Engineered + filtered features:", X_train_filtered.shape[1])
print("Removed:", len(columns_to_drop))

Engineered features: 125
Engineered + filtered features: 97
Removed: 28


### Preprocessing the Filtered Feature Set

To reduce redundancy, 28 housing-related `_MEDI` and `_MODE`
columns were removed, retaining the corresponding `_AVG` columns.
Each removed column had an absolute correlation above 0.95 with
its `_AVG` counterpart and an identical missing-value pattern
in the training data.

This reduced the input feature count from **125 to 97**.

The filtered features use the same preprocessing approach:
- Numeric missing values are replaced with training-set medians,
  with missing-value indicators added.
- Categorical missing values are labelled "Missing".
- Categorical features are one-hot encoded, with unseen categories ignored.

A separate preprocessor is fitted only on the training data and
then applied to validation and test data.

After preprocessing, all three datasets contain **264 features**
and **zero missing values**, compared with 320 features in the
unfiltered version.

The filtered and unfiltered versions will be compared using
validation performance. High correlation alone does not establish
that removing these features improves prediction.

In [16]:
from sklearn.base import clone

filtered_numeric_columns = X_train_filtered.select_dtypes(
    include="number"
).columns.tolist()

filtered_categorical_columns = X_train_filtered.select_dtypes(
    include="object"
).columns.tolist()

filtered_preprocessor = ColumnTransformer([
    ("numeric", clone(numeric_pipeline), filtered_numeric_columns),
    ("categorical", clone(categorical_pipeline), filtered_categorical_columns)
])


train_filtered_array = filtered_preprocessor.fit_transform(X_train_filtered)
valid_filtered_array = filtered_preprocessor.transform(X_valid_filtered)
test_filtered_array = filtered_preprocessor.transform(X_test_filtered)

filtered_feature_names = filtered_preprocessor.get_feature_names_out()

X_train_filtered_processed = pd.DataFrame(
    train_filtered_array,
    columns=filtered_feature_names,
    index=X_train_filtered.index
)

X_valid_filtered_processed = pd.DataFrame(
    valid_filtered_array,
    columns=filtered_feature_names,
    index=X_valid_filtered.index
)

X_test_filtered_processed = pd.DataFrame(
    test_filtered_array,
    columns=filtered_feature_names,
    index=X_test_filtered.index
)

for name, data in [
    ("Training", X_train_filtered_processed),
    ("Validation", X_valid_filtered_processed),
    ("Test", X_test_filtered_processed)
]:
    print(name, data.shape, "| Missing values:", data.isna().sum().sum())

Training (215256, 264) | Missing values: 0
Validation (46127, 264) | Missing values: 0
Test (46127, 264) | Missing values: 0


In [17]:
engineered_dir = PROCESSED_DIR / "engineered"
filtered_dir = PROCESSED_DIR / "engineered_filtered"

for folder, train, valid, test in [
    (
        engineered_dir,
        X_train_processed,
        X_valid_processed,
        X_test_processed
    ),
    (
        filtered_dir,
        X_train_filtered_processed,
        X_valid_filtered_processed,
        X_test_filtered_processed
    )
]:
    folder.mkdir(parents=True, exist_ok=True)

    train.to_csv(folder / "X_train.csv", index=False)
    valid.to_csv(folder / "X_valid.csv", index=False)
    test.to_csv(folder / "X_test.csv", index=False)

    y_train.to_csv(folder / "y_train.csv", index=False)
    y_valid.to_csv(folder / "y_valid.csv", index=False)
    y_test.to_csv(folder / "y_test.csv", index=False)

    train_ids.to_csv(folder / "train_ids.csv", index=False)
    valid_ids.to_csv(folder / "valid_ids.csv", index=False)
    test_ids.to_csv(folder / "test_ids.csv", index=False)

    print("Saved:", folder)

Saved: C:\Case Study Trees\HomeCreditStudy\data\processed\engineered
Saved: C:\Case Study Trees\HomeCreditStudy\data\processed\engineered_filtered


In [18]:
joblib.dump(
    preprocessor,
    ARTIFACT_DIR / "engineered_preprocessor.joblib"
)

joblib.dump(
    filtered_preprocessor,
    ARTIFACT_DIR / "filtered_preprocessor.joblib"
)

pd.Series(columns_to_drop, name="feature").to_csv(
    ARTIFACT_DIR / "filtered_columns.csv",
    index=False
)

print("Preprocessors and filtering list saved.")

Preprocessors and filtering list saved.
